# Reviewer-response experiments — W2, W3, W4

Self-contained. Runs on **Kaggle or Colab**. Safe to re-run after a disconnect.

| Weakness a reviewer would raise | Fixed by | Needs GPU |
|---|---|---|
| **W2** base rates are *estimated* under a Poisson assumption | `sensitivity.py` | no |
| **W3** only 6 models x 5 groups — is the correlation real? | `sensitivity.py` | no |
| **W4** the demonstration is one architecture at one depth | `threshold_sweep.py` | yes |

**W4 is the important one.** It shows the artifact with **no graph and no
architecture change at all** — only the decision threshold moves. AUC is
threshold-free so it is constant by construction; F1 traces a curve. If moving
one knob reproduces the graph model's entire "fairness improvement", then the
improvement was never a property of the model.

---
Set **GPU** on before running (Kaggle: Settings → Accelerator; Colab: Runtime → Change runtime type).

## 0. Config

In [ ]:
SEEDS  = 5
ROUNDS = 40
print(f'SEEDS={SEEDS}  ROUNDS={ROUNDS}')

## 1. Setup — works on Kaggle and Colab, rebuilds only what is missing

In [ ]:
import os, subprocess
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(BASE); print('working in', BASE)
os.makedirs('data', exist_ok=True); os.makedirs('results', exist_ok=True)

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n!!! NO GPU — turn on the accelerator, then re-run this cell.\n')

if not os.path.exists('data/la_crime.csv'):
    subprocess.run('rm -rf FedCrime && git clone --depth 1 -q '
                   'https://github.com/vanetlabiitj/FedCrime.git', shell=True)
    subprocess.run('cp FedCrime/Dataset/processed_crime.csv data/la_crime.csv', shell=True)
import pandas as pd
print('LA rows:', len(pd.read_csv('data/la_crime.csv')), '(expect 182325)')

## 2. Write the pipeline files

These cells **save** files, they do not run. Expect only `Writing ...` output.

In [ ]:
%%writefile robust_fair_gnn.py
"""
Robust & Fair Federated GNN for Crime Prediction under Extreme Sparsity
================================================================================
Research extension of FedCrime. Runs on the REAL Los Angeles / Chicago data.

It brings together, in ONE framework:
  * ZINB zero-inflation loss                       (from the FedCrime paper)
  * A Spatio-Temporal GNN (TCN + graph convolution) over a learned region graph
  * ATTACKS by malicious clients, incl. the novel *sparsity-camouflaged* attack
  * DEFENSES, incl. a novel density/graph "vouching" aggregator
  * FAIRNESS metrics (Head / Mid / Tail performance gap)
  * WEEK-AHEAD prediction (--horizon) and UNCERTAINTY (MC-dropout)

The research question (the gap): in federated crime prediction, honest SPARSE
neighbourhoods look like MALICIOUS clients, so standard defenses either let
attacks through or unfairly silence poor regions. We study this
sparsity-robustness-fairness trilemma and a defense that resolves it.

Examples
--------
# clean run on LA, next-day prediction, plain FedAvg:
python3 robust_fair_gnn.py --city la

# compare defenses under the sparsity-camouflaged attack (the key experiment):
python3 robust_fair_gnn.py --city la --attack camouflage --compare

# week-ahead (7 days) with uncertainty:
python3 robust_fair_gnn.py --city la --horizon 7 --mc 10

Requires: torch, numpy, pandas, scikit-learn
================================================================================
"""
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

EPS = 1e-10
C = 8                       # crime categories
L = 8                       # input days per window


def seed_everything(seed: int):
    """Make a run bit-reproducible across sessions and machines.

    torch.manual_seed alone is NOT enough on GPU: cuDNN picks algorithms by
    autotuning and several CUDA kernels accumulate non-deterministically, so
    the same seed can give different results between runs.  Without this,
    numbers cannot be reproduced for a paper.
    """
    import os, random
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:                       # older torch
        torch.use_deterministic_algorithms(True)

CITY = {
    "la":      ("data/la_crime.csv",  "date_occ", "2018-01-01", "2018-12-31"),
    "chicago": ("data/chi_crime.csv", "date_occ", "2015-01-01", "2015-12-31"),
    "nyc":     ("data/nyc_crime.csv", "date_occ", "2019-01-01", "2019-12-31"),
    "sf":      ("data/sf_crime.csv",  "date_occ", "2019-01-01", "2019-12-31"),
}


# --------------------------------------------------------------------------- #
# 1. REAL DATA -> aligned (days x regions x categories) tensor + region graph
# --------------------------------------------------------------------------- #
def load_city(csv, date_col, start, end):
    df = pd.read_csv(csv)
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])
    days = pd.date_range(start, end, freq="D")
    regions = sorted(df["neighborhood_id"].unique())
    ridx = {r: i for i, r in enumerate(regions)}
    didx = {d: i for i, d in enumerate(days)}
    mat = np.zeros((len(days), len(regions), C), dtype=np.float32)
    g = df.groupby([df[date_col].dt.normalize(), "neighborhood_id", "crime_type_id"]).size()
    for (day, r, c), _ in g.items():
        if day in didx and r in ridx and 0 <= int(c) < C:
            mat[didx[day], ridx[r], int(c)] = 1.0
    return mat, regions                        # mat: (D, R, C) binary


def make_windows(mat, horizon=1, dow=True):
    """X: (N, L, R, C[+2]), Y: (N, R, C) predicting `horizon` steps ahead.

    dow=True appends two day-of-week signals (sin/cos of the TARGET day) to every
    region's channel vector. Crime has strong weekly rhythms, so this gives the
    model genuine predictive signal beyond the trivial base rate.
    """
    D = mat.shape[0]
    X, Y = [], []
    for t in range(D - L - horizon + 1):
        win = mat[t:t + L]                       # (L, R, C)
        if dow:
            tgt = t + L + horizon - 1            # index of the day we predict
            ang = 2 * np.pi * (tgt % 7) / 7.0
            extra = np.zeros((L, win.shape[1], 2), np.float32)
            extra[:, :, 0] = np.sin(ang); extra[:, :, 1] = np.cos(ang)
            win = np.concatenate([win, extra], axis=2)   # (L, R, C+2)
        X.append(win)
        Y.append(mat[t + L + horizon - 1])
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)


def build_graph(mat_train, k=6):
    """Region graph from crime-pattern similarity on the TRAINING period."""
    totals = mat_train.sum(axis=2)             # (Dtrain, R) daily total per region
    R = totals.shape[1]
    corr = np.corrcoef(totals.T)               # (R, R)
    corr = np.nan_to_num(corr)
    np.fill_diagonal(corr, -1)
    A = np.eye(R, dtype=np.float32)
    for i in range(R):                         # connect each region to top-k similar
        for j in np.argsort(corr[i])[::-1][:k]:
            A[i, j] = 1.0
    A = np.maximum(A, A.T)                      # symmetric
    d = A.sum(1); dinv = 1.0 / np.sqrt(np.maximum(d, 1e-6))
    return (A * dinv[:, None] * dinv[None, :]).astype(np.float32), A


def head_mid_tail(mat_train):
    """Label each region Head/Mid/Tail by total training crime (20/30/50%)."""
    totals = mat_train.sum(axis=(0, 2))        # (R,)
    order = np.argsort(totals)[::-1]
    R = len(totals); nh = max(1, round(R * 0.2)); nm = max(1, round(R * 0.3))
    tag = np.empty(R, dtype=object)
    tag[order[:nh]] = "Head"; tag[order[nh:nh + nm]] = "Mid"; tag[order[nh + nm:]] = "Tail"
    return tag


# --------------------------------------------------------------------------- #
# 2. MODEL: ST-GNN (TCN temporal + graph conv) + ZINB + classifier + dropout
# --------------------------------------------------------------------------- #
class TCNBlock(nn.Module):
    def __init__(self, i, o, k=3, dil=1):
        super().__init__()
        p = (k - 1) * dil // 2
        self.c1 = nn.Conv1d(i, o, k, padding=p, dilation=dil)
        self.c2 = nn.Conv1d(o, o, k, padding=p, dilation=dil)
        self.r = nn.ReLU()
        self.res = nn.Conv1d(i, o, 1) if i != o else nn.Identity()

    def forward(self, x):
        s = self.res(x); x = self.r(self.c1(x)); x = self.r(self.c2(x)); return x + s


class STGNN(nn.Module):
    """gnn_type: 'plain' (basic GCN), 'gated' (fixes over-smoothing),
    'attention', or 'ode' (continuous-depth proxy; see `_ode`)."""
    def __init__(self, hidden=16, use_graph=True, gnn_type="plain", p_drop=0.2,
                 in_ch=None, gnn_layers=2, alpha=0.25, ode_source=False):
        super().__init__()
        self.use_graph = use_graph
        self.gnn_type = gnn_type
        self.alpha = float(alpha)          # ODE step size (integration time = alpha*n_layers)
        self.ode_source = bool(ode_source)  # CGNN-style h0 injection
        self.in_ch = in_ch if in_ch is not None else C
        self.tcn = nn.Sequential(TCNBlock(self.in_ch, hidden, dil=1),
                                 TCNBlock(hidden, hidden, dil=2),
                                 TCNBlock(hidden, hidden, dil=4))
        # graph depth is configurable: over-smoothing is expected to worsen
        # with depth, so this lets us test the mechanism directly.
        self.n_layers = max(1, int(gnn_layers))
        self.glin = nn.ModuleList([nn.Linear(hidden, hidden)
                                   for _ in range(self.n_layers)])
        self.glate = nn.ModuleList([nn.Linear(hidden, hidden)
                                    for _ in range(self.n_layers)])
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p_drop)
        self.fc_pi = nn.Linear(hidden, C)      # Eq 6
        self.fc_mu = nn.Linear(hidden, C)      # Eq 7
        self.fc_phi = nn.Linear(hidden, C)     # Eq 8
        self.fc_out = nn.Linear(hidden, C)     # crime-present classifier

    def _gated(self, h, A, lin, gate):
        # gated residual: g=sigmoid(gate(h)); H = g*(A.H.W) + (1-g)*H
        # each region learns how much to trust neighbours vs its own signal
        agg = torch.einsum('ij,bjh->bih', A, lin(h))
        g = torch.sigmoid(gate(h))
        return self.relu(g * agg + (1 - g) * h)

    def _attn(self, h, A, lin):
        # attention over graph neighbours (masked by the adjacency)
        Wh = lin(h); Hd = Wh.shape[-1]
        scores = torch.einsum('bih,bjh->bij', Wh, Wh) / (Hd ** 0.5)
        mask = (A > 0).float().unsqueeze(0)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        alpha = torch.softmax(scores, dim=-1)
        return self.relu(torch.einsum('bij,bjh->bih', alpha, Wh))

    def _ode(self, h, A):
        """Continuous-depth (Neural ODE) proxy via fixed-step forward Euler.

        Instead of stacking K discrete graph layers, integrate a single vector
        field for K steps of size alpha:

            dh/dt = f(h, A) = sigma(A . h . W) - h      [+ h0 if ode_source]
            h <- h + alpha * f(h, A)

        Two details make this a genuine ODE discretisation rather than just a
        residual GCN:
          1. ONE shared weight matrix (self.glin[0]) is reused at every step --
             an ODE has a single f, not a different W per layer.
          2. The -h decay term makes the dynamics a relaxation toward the
             aggregated neighbour state, so the system is well-posed rather
             than accumulating without bound.

        Effective integration time is alpha * n_layers, so small alpha with
        many steps == "pouring the neighbour signal in slowly".  With
        ode_source=True the initial state h0 is re-injected at every step,
        which is the mechanism CGNN uses to resist over-smoothing.

        Refs: Chen et al. 2018 (Neural ODEs); Poli et al. 2019 (GDE);
              Xhonneux et al. 2020 (CGNN).
        """
        lin, h0 = self.glin[0], h
        for _ in range(self.n_layers):
            agg = self.relu(torch.einsum('ij,bjh->bih', A, lin(h)))
            f = agg - h + (h0 if self.ode_source else 0.0)
            h = h + self.alpha * f
        return h

    def _mix(self, h, A):
        # NOTE: the no-graph control uses the SAME number of dense layers, so
        # any difference is attributable to the graph, not to depth/capacity.
        if not self.use_graph:                       # no graph (TCN only)
            for lin in self.glin:
                h = self.relu(lin(h))
            return h
        if self.gnn_type == "ode":                   # continuous-depth proxy
            return self._ode(h, A)
        for i, lin in enumerate(self.glin):
            if self.gnn_type == "gated":
                h = self._gated(h, A, lin, self.glate[i])
            elif self.gnn_type == "attention":
                h = self._attn(h, A, lin)
            else:                                    # plain GCN
                h = self.relu(torch.einsum('ij,bjh->bih', A, lin(h)))
        return h

    def forward(self, X, A):
        B, Lw, R, Ch = X.shape
        h = X.permute(0, 2, 3, 1).reshape(B * R, Ch, Lw)
        h = self.tcn(h)[:, :, -1].reshape(B, R, -1)
        h = self.drop(self._mix(h, A))
        pi = torch.sigmoid(self.fc_pi(h))
        mu = torch.exp(torch.clamp(self.fc_mu(h), max=15.0))
        phi = torch.nn.functional.softplus(self.fc_phi(h))
        return pi, mu, phi, self.fc_out(h)


def zinb_elem(pi, mu, phi, y):
    """Per-element ZINB negative log-likelihood, shape (B, R, C)."""
    is0 = y.eq(0).float(); is1 = y.gt(0).float()
    zero = is0 * torch.log(pi + EPS)
    nb = is1 * (torch.lgamma(y + phi) - torch.lgamma(phi) - torch.lgamma(y + 1.0)
                + phi * (torch.log(phi + EPS) - torch.log(phi + mu + EPS))
                + y * (torch.log(mu + EPS) - torch.log(phi + mu + EPS)))
    return -(zero + nb)


def zinb_loss(pi, mu, phi, y):
    return zinb_elem(pi, mu, phi, y).mean()


def loss_fn(pi, mu, phi, logit, y, pos_weight=None, region_weight=None,
            group_ids=None, dro_tau=0.0, lam=0.3):
    # pos_weight counters class imbalance. Group-DRO (dro_tau>0) adaptively
    # up-weights the WORST-performing group (the poor tail regions) each step,
    # so training focuses on closing the fairness gap instead of ignoring the
    # tail. region_weight is the older static fairness weighting.
    bce = torch.nn.functional.binary_cross_entropy_with_logits(
        logit, y, pos_weight=pos_weight, reduction="none")     # (B, R, C)
    loss = bce + lam * zinb_elem(pi, mu, phi, y)               # (B, R, C)
    if group_ids is not None and dro_tau > 0:
        lr = loss.mean(dim=(0, 2))                             # per-region loss (R,)
        groups = torch.unique(group_ids)
        gloss = torch.stack([lr[group_ids == g].mean() for g in groups])
        gw = torch.softmax(dro_tau * gloss, dim=0) * len(groups)   # worse -> higher
        rw = torch.ones_like(lr)
        for i, g in enumerate(groups):
            rw[group_ids == g] = gw[i]
        loss = loss * rw.view(1, -1, 1)
    elif region_weight is not None:
        loss = loss * region_weight.view(1, -1, 1)
    return loss.mean()


# --------------------------------------------------------------------------- #
# 3. ATTACKS (applied inside a malicious client)
# --------------------------------------------------------------------------- #
def poison_labels(Y, attack):
    if attack == "labelflip":
        return 1.0 - Y                          # flip yes<->no
    if attack == "camouflage":
        return np.zeros_like(Y)                 # pretend "no crime anywhere" (looks sparse)
    return Y


def poison_update(delta, attack, factor=8.0):
    if attack == "scale":
        return [d * factor for d in delta]      # blow up the update
    if attack == "camouflage":
        return [d * 0.3 for d in delta]         # small, sparse-looking update
    return delta


# --------------------------------------------------------------------------- #
# 4. FEDERATED TRAINING with pluggable DEFENSE
# --------------------------------------------------------------------------- #
def gw(net): return [p.detach().clone() for p in net.state_dict().values()]
def sw(net, w):
    sd = net.state_dict()
    for k, v in zip(sd.keys(), w): sd[k] = v.clone()
    net.load_state_dict(sd)
def flat(w): return torch.cat([t.flatten() for t in w])


def aggregate(defense, wsets, gwt, densities):
    """Combine client weight-sets into a new global model."""
    n = len(wsets)
    if defense == "fedavg":
        return [torch.stack([w[i] for w in wsets]).mean(0) for i in range(len(wsets[0]))]

    deltas = [[w[i] - gwt[i] for i in range(len(gwt))] for w in wsets]

    if defense == "strust2":
        # STRUST-v2: norm-clip to the median norm (kills scaling attacks
        # deterministically), then trust-weight by direction agreement with a
        # coordinate-wise MEDIAN reference (robust to a minority of attackers),
        # then aggregate. More stable than v1 across attacks/cities.
        flatd = [flat(d) for d in deltas]
        norms = torch.tensor([float(f.norm()) + 1e-9 for f in flatd])
        med = float(norms.median())
        clip = torch.clamp(med / norms, max=1.0)               # scale down only
        stacked = torch.stack([flatd[i] * clip[i] for i in range(n)])
        ref = stacked.median(0).values                          # robust reference
        ref = ref / (ref.norm() + 1e-9)
        cos = torch.stack([(stacked[i] / (stacked[i].norm() + 1e-9)) @ ref
                           for i in range(n)])
        trust = torch.clamp(cos, min=0.0) ** 2                  # sharpen
        if float(trust.sum()) < 1e-6:
            trust = torch.ones(n)
        trust = trust / trust.sum()
        out = []
        for li in range(len(gwt)):
            acc = sum(trust[i] * deltas[i][li] * clip[i] for i in range(n))
            out.append(gwt[li] + acc)
        return out

    if defense == "strust":
        # SPARSITY-AWARE TRUST  (our trilemma solution).
        # Standard defenses reject "outlier-distant" clients -> they wrongly
        # reject honest sparse (tail) regions. Instead we judge clients by the
        # DIRECTION of their update and neutralise magnitude:
        #   1) unit-normalise each update  -> a scaling attack becomes harmless
        #   2) trust = max(0, cos(update, robust median direction))
        #        -> label-flip / camouflage point the wrong way  -> trust 0
        #        -> honest sparse clients point the right way     -> trust kept
        #   3) trust-weighted average, rescaled to the typical honest magnitude.
        flatd = [flat(d) for d in deltas]
        norms = [float(f.norm()) + 1e-9 for f in flatd]
        units = torch.stack([flatd[i] / norms[i] for i in range(n)])
        ref = units.median(0).values
        ref = ref / (ref.norm() + 1e-9)
        trust = torch.clamp(units @ ref, min=0.0)              # direction agreement
        if float(trust.sum()) < 1e-6:
            trust = torch.ones(n)
        trust = trust / trust.sum()
        scale = float(torch.tensor(norms).median())
        out = []
        for li in range(len(gwt)):
            acc = sum(trust[i] * (deltas[i][li] / norms[i]) for i in range(n))
            out.append(gwt[li] + scale * acc)
        return out

    F = torch.stack([flat(d) for d in deltas])                 # (n, P)
    dist = torch.cdist(F, F)                                    # pairwise distances

    if defense == "trimmed":
        keep = max(1, n - 2)
        idx = torch.argsort(dist.sum(1))[:keep]
    elif defense == "krum":
        kk = max(1, n - 2)
        scores = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        idx = torch.argsort(scores)[:max(1, n - 2)]
    elif defense == "vouch":
        # NOVEL: a client's "oddness" is EXPECTED to be high if it is sparse
        # (honest tail region). Divide the Krum score by the client's data
        # density so sparse-honest clients are NOT penalised, while dense-yet-
        # odd clients (real attackers) are. Then keep the low-suspicion clients.
        kk = max(1, n - 2)
        raw = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        dens = torch.tensor(densities, dtype=raw.dtype).clamp(min=1e-3)
        suspicion = raw * dens                                 # density-adjusted
        idx = torch.argsort(suspicion)[:max(1, n - 2)]
    else:
        raise ValueError(defense)

    sel = [wsets[i] for i in idx.tolist()]
    return [torch.stack([w[i] for w in sel]).mean(0) for i in range(len(sel[0]))]


def train(clients, A_full, defense, attack, mal_ids, use_graph=True,
          rounds=60, local_epochs=5, lr=0.03, seed=0, pos_weight=None,
          gnn_type="plain", fair=False, dro_tau=0.0, in_ch=None,
          gnn_layers=2, alpha=0.25, ode_source=False, clip=1.0):
    # FULL determinism.  Seeding only torch's CPU generator left cuDNN
    # autotuning and non-deterministic CUDA kernels free to vary run to run,
    # which made results irreproducible across sessions.
    seed_everything(seed)
    mk = lambda: STGNN(use_graph=use_graph, gnn_type=gnn_type, in_ch=in_ch,
                       gnn_layers=gnn_layers, alpha=alpha, ode_source=ode_source)
    g = mk(); gwt = gw(g)
    for _ in range(rounds):
        wsets, dens = [], []
        for ci, cl in enumerate(clients):
            local = mk(); sw(local, gwt)
            A = torch.tensor(cl["A"]); X = torch.tensor(cl["X"])
            Y = torch.tensor(poison_labels(cl["Y"], attack if ci in mal_ids else "none"))
            opt = torch.optim.Adam(local.parameters(), lr=lr, weight_decay=1e-4)
            rw = cl.get("rw") if fair else None
            grp = cl.get("grp")
            local.train()
            for _ in range(local_epochs):
                opt.zero_grad()
                pi, mu, phi, logit = local(X, A)
                loss = loss_fn(pi, mu, phi, logit, Y, pos_weight=pos_weight,
                               region_weight=rw, group_ids=grp, dro_tau=dro_tau)
                if not torch.isfinite(loss):
                    # deep graph stacks can diverge; skip the step rather than
                    # let NaNs propagate into the global model silently
                    opt.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(local.parameters(), clip)
                opt.step()
            delta = [p - q for p, q in zip(gw(local), gwt)]
            if ci in mal_ids:
                delta = poison_update(delta, attack)
            wsets.append([q + d for q, d in zip(gwt, delta)])
            dens.append(float(cl["Y"].mean()))                 # client data density
        gwt = aggregate(defense, wsets, gwt, dens)
        sw(g, gwt)
    return g


# --------------------------------------------------------------------------- #
# 5. EVALUATION: overall + fairness (Head/Mid/Tail) + uncertainty
# --------------------------------------------------------------------------- #
@torch.no_grad()
def tune_thresholds(net, X, Y, A):
    """Per-(region, crime-type) decision threshold tuned on a VALIDATION set to
    maximise each cell's F1. Rare crimes in poor regions get a LOWER threshold
    so they are actually predicted -> higher tail F1 -> smaller fairness gap.
    This is the fairness fix: it targets the decision, not the loss."""
    net.eval()
    prob = torch.sigmoid(net(torch.tensor(X), torch.tensor(A))[3]).numpy()  # (N,R,C)
    R = Y.shape[1]
    grid = np.arange(0.03, 0.60, 0.02)
    thr = np.full((R, C), 0.5, dtype=np.float32)
    for r in range(R):
        for cc in range(C):
            y = Y[:, r, cc]
            if y.sum() == 0:                     # no positives to tune on
                continue
            p = prob[:, r, cc]
            best_t, best_f = 0.5, -1.0
            for t in grid:
                f = f1_score(y, (p > t).astype(float), zero_division=0)
                if f > best_f:
                    best_f, best_t = f, t
            thr[r, cc] = best_t
    return thr


@torch.no_grad()
def evaluate(net, X, Y, A, tags, mc=0, thr=None):
    net.eval()
    Xt = torch.tensor(X); At = torch.tensor(A)
    if mc > 0:                                   # MC-dropout uncertainty
        net.train()                              # keep dropout ON
        probs = torch.stack([torch.sigmoid(net(Xt, At)[3]) for _ in range(mc)])
        prob = probs.mean(0); uncertainty = probs.std(0).mean().item()
        net.eval()
    else:
        prob = torch.sigmoid(net(Xt, At)[3]); uncertainty = float("nan")
    prob = torch.nan_to_num(prob, nan=0.0, posinf=1.0, neginf=0.0)
    if thr is not None:                          # adaptive threshold
        tt = torch.tensor(thr, dtype=prob.dtype)
        th = tt.view(1, -1, 1) if tt.dim() == 1 else tt.unsqueeze(0)  # (1,R[,C])
        pred = (prob > th).float().numpy()
    else:
        pred = (prob > 0.5).float().numpy()      # (N, R, C)
    N, R, _ = pred.shape

    def group_f1(mask):
        yy = Y[:, mask, :].reshape(-1, C); pp = pred[:, mask, :].reshape(-1, C)
        return f1_score(yy, pp, average="macro", zero_division=0) * 100

    def group_ceiling(mask):
        """Max attainable macro-F1 for this group's base rates.

        For a label with positive rate p, precision <= p and recall <= 1, so
        F1 <= 2p/(1+p). This ceiling SCALES WITH BASE RATE, which is why sparse
        (tail) regions can never reach head-level raw F1 however good the model.
        """
        yg = Y[:, mask, :]
        p = yg.mean(axis=(0, 1))                       # per-category base rate
        return float((2 * p / (1 + p)).mean()) * 100

    overall = f1_score(Y.reshape(-1, C), pred.reshape(-1, C), average="macro", zero_division=0) * 100
    res = {"overall": overall, "uncertainty": uncertainty}

    # ---- PER-GROUP metric decomposition (tests the theory) ---------------- #
    # Base-rate DEPENDENT metrics (F1, precision, accuracy) should show a large
    # Head-Tail gap; base-rate INVARIANT metrics (AUC, balanced accuracy, TPR)
    # should show almost none -- on the SAME predictions.
    probn = np.nan_to_num(prob.numpy(), nan=0.0, posinf=1.0, neginf=0.0)
    for grp in ["Head", "Mid", "Tail"]:
        m = np.array([t == grp for t in tags])
        yg = Y[:, m, :].reshape(-1, C)
        pg = pred[:, m, :].reshape(-1, C)
        sg = probn[:, m, :].reshape(-1, C)
        f1s, precs, accs, aucs_g, bals, tprs = [], [], [], [], [], []
        for cc_ in range(C):
            y, p_, s_ = yg[:, cc_], pg[:, cc_], sg[:, cc_]
            if y.sum() == 0 or y.sum() == len(y):
                continue
            tp = ((p_ == 1) & (y == 1)).sum(); fp = ((p_ == 1) & (y == 0)).sum()
            fn = ((p_ == 0) & (y == 1)).sum(); tn = ((p_ == 0) & (y == 0)).sum()
            f1s.append(0.0 if (2*tp+fp+fn) == 0 else 2*tp/(2*tp+fp+fn))
            precs.append(0.0 if (tp+fp) == 0 else tp/(tp+fp))
            accs.append((tp+tn)/len(y))
            tpr = 0.0 if (tp+fn) == 0 else tp/(tp+fn)
            tnr = 0.0 if (tn+fp) == 0 else tn/(tn+fp)
            tprs.append(tpr); bals.append(0.5*(tpr+tnr))
            aucs_g.append(0.5 if np.allclose(s_, s_[0]) else roc_auc_score(y, s_))
        mean = lambda v: float(np.mean(v))*100 if v else float("nan")
        res[grp+"_f1"] = mean(f1s); res[grp+"_prec"] = mean(precs)
        res[grp+"_acc"] = mean(accs); res[grp+"_auc"] = mean(aucs_g)
        res[grp+"_bal"] = mean(bals); res[grp+"_tpr"] = mean(tprs)
    for k in ["f1", "prec", "acc", "auc", "bal", "tpr"]:
        res["gap_"+k] = res.get("Head_"+k, float("nan")) - res.get("Tail_"+k, float("nan"))

    # ---- REAL SKILL: does the model beat trivial baselines? ----------------
    pr = prob.numpy().reshape(-1, C); yy = Y.reshape(-1, C)
    # a collapsed/attacked model can emit NaN or inf -> treat as no-skill (0.5)
    pr = np.nan_to_num(pr, nan=0.0, posinf=1.0, neginf=0.0)
    aucs, aps, lifts = [], [], []
    for cc_ in range(C):
        yc, pc = yy[:, cc_], pr[:, cc_]
        if 0 < yc.sum() < len(yc):
            if np.allclose(pc, pc[0]):        # constant scores -> no ranking skill
                aucs.append(0.5); aps.append(yc.mean())
                lifts.append(1.0)
                continue
            aucs.append(roc_auc_score(yc, pc))                 # 0.5 = no skill
            ap = average_precision_score(yc, pc)
            aps.append(ap)
            lifts.append(ap / max(yc.mean(), 1e-9))            # AP vs random(=base rate)
    res["auc"] = float(np.mean(aucs)) if aucs else float("nan")
    res["ap_lift"] = float(np.mean(lifts)) if lifts else float("nan")
    # trivial "always predict crime" baseline F1 (macro)
    p_cat = yy.mean(0)
    res["baseline_f1"] = float((2 * p_cat / (1 + p_cat)).mean()) * 100
    res["f1_over_baseline"] = overall - res["baseline_f1"]
    for grp in ["Head", "Mid", "Tail"]:
        m = np.array([t == grp for t in tags])
        res[grp] = group_f1(m)
        ceil = group_ceiling(m)
        # SKILL = how much of the attainable performance the model actually
        # achieves (base-rate-normalised). This is the fair way to compare
        # groups whose ceilings differ.
        res[grp + "_skill"] = 100.0 * res[grp] / max(ceil, 1e-6)
    res["fairness_gap"] = res["Head"] - res["Tail"]          # raw (base-rate biased)
    res["skill_gap"] = res["Head_skill"] - res["Tail_skill"] # normalised (fair)
    return res


# --------------------------------------------------------------------------- #
# 6. RUN
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", default="la", choices=list(CITY))
    ap.add_argument("--horizon", type=int, default=1, help="1=next day, 7=week ahead")
    ap.add_argument("--clients", type=int, default=6)
    ap.add_argument("--rounds", type=int, default=60)
    ap.add_argument("--attack", default="none",
                    choices=["none", "labelflip", "scale", "camouflage"])
    ap.add_argument("--attack-frac", type=float, default=0.25)
    ap.add_argument("--defense", default="fedavg",
                    choices=["fedavg","trimmed","krum","vouch","strust","strust2"])
    ap.add_argument("--mc", type=int, default=0, help="MC-dropout passes (0=off)")
    ap.add_argument("--fair", action="store_true",
                    help="up-weight poor (tail) regions in training for fairness")
    ap.add_argument("--adathr", action="store_true",
                    help="per-region adaptive thresholds (fairness fix)")
    ap.add_argument("--metrics", action="store_true",
                    help="metric-taxonomy experiment (per-group dependent vs invariant)")
    ap.add_argument("--dro", action="store_true",
                    help="Group-DRO: adaptively focus training on the worst group")
    ap.add_argument("--dro-tau", type=float, default=3.0,
                    help="Group-DRO strength (higher = more focus on the tail)")
    ap.add_argument("--compare", action="store_true",
                    help="run all defenses under the chosen attack")
    ap.add_argument("--graph-compare", action="store_true",
                    help="compare GNN vs no-GNN on this real city (clean, FedAvg)")
    ap.add_argument("--gnn-layers", type=int, default=2,
                    help="number of graph layers (tests over-smoothing vs depth)")
    ap.add_argument("--depth-sweep", action="store_true",
                    help="sweep graph depth 1..4 vs the no-graph control")
    ap.add_argument("--gnn", default="plain",
                    choices=["plain", "gated", "attention", "ode"],
                    help="graph type: gated/attention/ode fix over-smoothing")
    ap.add_argument("--alpha", type=float, default=0.25,
                    help="ODE step size (--gnn ode); integration time = alpha*layers")
    ap.add_argument("--ode-source", action="store_true",
                    help="CGNN-style h0 re-injection at every ODE step")
    ap.add_argument("--repro-check", action="store_true",
                    help="run the same config twice; verify bit-identical output")
    ap.add_argument("--ode-sweep", action="store_true",
                    help="sweep ODE step size alpha vs the no-graph control")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--save", default=None,
                    help="append results as JSON lines for the paper tables")
    ap.add_argument("--seeds", type=int, default=1,
                    help="run N seeds and report mean +/- std with a t-test")
    args = ap.parse_args()

    csv, dcol, start, end = CITY[args.city]
    print(f"Loading {args.city.upper()} from {csv} ...")
    mat, regions = load_city(csv, dcol, start, end)
    D, R, _ = mat.shape
    print(f"{R} regions, {C} crime types, {D} days, "
          f"sparsity {100*(1-mat.mean()):.1f}% zeros | horizon = {args.horizon} day(s)")

    X, Y = make_windows(mat, horizon=args.horizon)
    IN = X.shape[-1]        # input channels (C + day-of-week features)
    n = len(X); ntr = int(n * 0.7); nval = int(n * 0.1)
    Xtr, Ytr = X[:ntr], Y[:ntr]
    Xval, Yval = X[ntr:ntr + nval], Y[ntr:ntr + nval]     # for threshold tuning
    Xte, Yte = X[ntr + nval:], Y[ntr + nval:]
    A_full, _ = build_graph(mat[:ntr + L])
    tags = head_mid_tail(mat[:ntr + L])
    print(f"regions -> Head {sum(t=='Head' for t in tags)}  "
          f"Mid {sum(t=='Mid' for t in tags)}  Tail {sum(t=='Tail' for t in tags)}")

    # split regions across federated clients (round-robin by crime rank -> mixed)
    order = np.argsort(mat[:ntr].sum(axis=(0, 2)))[::-1]
    parts = [order[i::args.clients] for i in range(args.clients)]
    # fairness weights: sparse (tail) regions get higher weight (clipped 1..4)
    dens_r = Ytr.mean(axis=(0, 2))                       # per-region positive rate
    rw_global = np.clip(dens_r.mean() / (dens_r + 1e-6), 1.0, 4.0).astype(np.float32)
    gmap = {"Head": 0, "Mid": 1, "Tail": 2}
    grp_global = np.array([gmap[t] for t in tags], dtype=np.int64)   # for Group-DRO
    clients = [{"A": A_full[np.ix_(idx, idx)],
                "X": Xtr[:, :, idx, :], "Y": Ytr[:, idx, :],
                "rw": torch.tensor(rw_global[idx]),
                "grp": torch.tensor(grp_global[idx])} for idx in parts]
    n_mal = round(args.attack_frac * args.clients)
    mal_ids = set(range(n_mal))                 # first few clients are malicious
    if args.attack != "none":
        print(f"ATTACK: {args.attack} on {n_mal}/{args.clients} clients\n")

    # per-category class weights to fight the 68% zero imbalance
    yflat = Ytr.reshape(-1, C)
    pos = yflat.sum(0); neg = len(yflat) - pos
    pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, 10.0),
                              dtype=torch.float32)

    # ---- REPRO CHECK: is a run bit-reproducible? -------------------------- #
    # Gate every other experiment on this.  If the same seed and config give
    # two different answers, no table produced by this script can be trusted.
    if args.repro_check:
        print(f"\nREPRODUCIBILITY CHECK — {args.city.upper()} / {args.gnn}, "
              f"seed {args.seed}, {args.gnn_layers} layers, {args.rounds} rounds")
        outs = []
        for rep in (1, 2):
            net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                        use_graph=(args.gnn_layers > 0), rounds=args.rounds,
                        seed=args.seed, pos_weight=pos_weight,
                        gnn_type=args.gnn, in_ch=IN,
                        gnn_layers=max(1, args.gnn_layers),
                        alpha=args.alpha, ode_source=args.ode_source)
            r = evaluate(net, Xte, Yte, A_full, tags)
            outs.append(r)
            print(f"  run {rep}: overall {r['overall']:.6f}  "
                  f"Head {r['Head']:.6f}  Tail {r['Tail']:.6f}")
        d = abs(outs[0]["overall"] - outs[1]["overall"])
        print(f"\n  |difference| = {d:.8f}")
        print("  PASS — runs are bit-identical; results are reproducible."
              if d < 1e-9 else
              "  FAIL — same seed gave different answers. Do NOT trust any\n"
              "         table from this script until this is resolved.")
        return

    # ---- ODE SWEEP: does continuous depth rescue the graph? --------------- #
    # Motivation: the depth sweep showed plain/attention degrade with depth
    # while gated stays flat AT the no-graph level.  Continuous-depth models
    # (GDE, CGNN) are proposed as an over-smoothing fix, so a reviewer will
    # ask.  If every alpha lands on the no-graph control, the limitation is
    # informational (neighbours carry no signal), not architectural.
    if args.ode_sweep:
        seeds_o = list(range(args.seed, args.seed + max(1, args.seeds)))
        alphas = [0.05, 0.1, 0.25, 0.5, 1.0]
        src = " +h0" if args.ode_source else ""
        print(f"\nODE STEP-SIZE SWEEP — {args.city.upper()} / ode{src} "
              f"({len(seeds_o)} seeds, {args.rounds} rounds, "
              f"{args.gnn_layers} steps)")
        print(f"{'alpha':>8s} | {'t=a*K':>6s} | {'Overall':>13s} | "
              f"{'Head':>6s} {'Tail':>13s} | {'RawGap':>7s} | {'SkillGap':>8s}")
        print("-" * 80)
        # TWO controls, because parameter counts differ in BOTH directions:
        #   * the ODE reuses ONE weight matrix K times  -> fewer params
        #   * a K-dense-layer no-graph net has K matrices -> more params
        # Reporting both brackets the capacity range, so a difference that
        # survives BOTH controls is attributable to the graph.
        for a in [None, "ctrl1"] + alphas:
            ug = a is not None and a != "ctrl1"
            acc = {k: [] for k in ("overall", "Head", "Tail",
                                   "fairness_gap", "skill_gap")}
            for sd in seeds_o:
                net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                            use_graph=ug, rounds=args.rounds, seed=sd,
                            pos_weight=pos_weight,
                            gnn_type=("ode" if ug else args.gnn),
                            in_ch=IN,
                            gnn_layers=(1 if a == "ctrl1" else args.gnn_layers),
                            alpha=(a if ug else 0.0),
                            ode_source=args.ode_source)
                th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
                r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
                for k in acc: acc[k].append(r[k])
            m = {k: float(np.mean(v)) for k, v in acc.items()}
            sdv = {k: float(np.std(v)) for k, v in acc.items()}
            lbl = ("no-gr/K" if a is None else
                   "no-gr/1" if a == "ctrl1" else f"{a:.2f}")
            tt = "-" if not ug else f"{a*args.gnn_layers:.2f}"
            print(f"{lbl:>8s} | {tt:>6s} | {m['overall']:6.2f}±{sdv['overall']:5.2f} | "
                  f"{m['Head']:6.2f} {m['Tail']:6.2f}±{sdv['Tail']:5.2f} "
                  f"| {m['fairness_gap']:7.2f} | {m['skill_gap']:8.2f}")
            if args.save:
                import json
                with open(args.save, "a") as fh:
                    fh.write(json.dumps({
                        "city": args.city, "mode": "ode_sweep",
                        "gnn": "ode" if ug else "none",
                        "alpha": (a if ug else None), "ode_source": bool(args.ode_source),
                        "steps": args.gnn_layers,
                        "integration_time": (a * args.gnn_layers if ug else None),
                        "seeds": len(seeds_o), "rounds": args.rounds,
                        "adathr": bool(args.adathr),
                        "overall_mean": m["overall"], "overall_std": sdv["overall"],
                        "Head": m["Head"], "Tail": m["Tail"],
                        "tail_std": sdv["Tail"], "raw_gap": m["fairness_gap"],
                        "skill_gap": m["skill_gap"]}) + "\n")
        print("\nTWO controls: no-gr/K has K dense layers, no-gr/1 has one.")
        print("The ODE ties ONE weight matrix across K steps, so its parameter")
        print("count sits BELOW no-gr/K and at no-gr/1.  An alpha must beat BOTH")
        print("controls before it counts as the graph helping.")
        print("NOTE: alpha=1.0 is NOT the depth-sweep plain GCN -- the ODE shares")
        print("one weight matrix across steps, the plain GCN uses K distinct ones.")
        print("alpha -> 0 recovers the no-graph control (no propagation);")
        print("large alpha approaches the discrete GCN.  If NO alpha beats the")
        print("no-graph row, continuous depth does not rescue the graph, and the")
        print("limitation is informational rather than architectural.")
        return

    # --- GNN vs no-GNN comparison on the real city (clean, FedAvg) ---
    # ---- DEPTH SWEEP: does over-smoothing worsen with graph depth? -------- #
    if args.depth_sweep:
        seeds_d = list(range(args.seed, args.seed + max(1, args.seeds)))
        # CAPACITY-MATCHED CONTROL.  An earlier version compared every graph
        # depth against a SINGLE no-graph run at 1 dense layer, which confounds
        # "the graph hurts" with "this model has fewer layers".  We now pair
        # each depth with its OWN no-graph control at the same layer count, so
        # the delta isolates the graph.
        print(f"\nGRAPH-DEPTH SWEEP — {args.city.upper()} / {args.gnn} "
              f"({len(seeds_d)} seeds, {args.rounds} rounds)")
        print("each depth is paired with a no-graph control at the SAME depth")
        print(f"{'layers':>6s} | {'graph F1':>13s} | {'no-graph F1':>13s} | "
              f"{'delta':>6s} | {'Tail g':>13s} | {'Tail n':>13s}")
        print("-" * 84)

        KEYS = ("overall", "Head", "Mid", "Tail", "fairness_gap", "skill_gap",
                "Head_auc", "Mid_auc", "Tail_auc", "gap_auc",
                "Head_tpr", "Tail_tpr", "gap_tpr",
                "Head_bal", "Tail_bal", "gap_bal",
                "Head_prec", "Tail_prec")

        def _run(ug, nl):
            acc = {k: [] for k in KEYS}
            for sd in seeds_d:
                net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                            use_graph=ug, rounds=args.rounds, seed=sd,
                            pos_weight=pos_weight, gnn_type=args.gnn,
                            in_ch=IN, gnn_layers=nl, alpha=args.alpha,
                            ode_source=args.ode_source)
                th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
                r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
                for k in acc:
                    acc[k].append(r.get(k, float("nan")))
            return ({k: float(np.nanmean(v)) for k, v in acc.items()},
                    {k: float(np.nanstd(v)) for k, v in acc.items()})

        for depth in [1, 2, 3, 4]:
            mg, sg = _run(True, depth)       # graph ON  at this depth
            mn, sn = _run(False, depth)      # graph OFF at this depth (control)
            print(f"{depth:>6d} | {mg['overall']:6.2f}±{sg['overall']:5.2f} | "
                  f"{mn['overall']:6.2f}±{sn['overall']:5.2f} | "
                  f"{mg['overall']-mn['overall']:+6.2f} | "
                  f"{mg['Tail']:6.2f}±{sg['Tail']:5.2f} | "
                  f"{mn['Tail']:6.2f}±{sn['Tail']:5.2f}")
            if args.save:
                import json
                with open(args.save, "a") as fh:
                    for tag, m, sdv, ug in (("graph", mg, sg, True),
                                            ("control", mn, sn, False)):
                        fh.write(json.dumps({
                            "city": args.city, "mode": "depth_sweep_paired",
                            "arm": tag, "use_graph": ug,
                            "gnn": args.gnn, "graph_layers": depth,
                            "seeds": len(seeds_d), "rounds": args.rounds,
                            "adathr": bool(args.adathr),
                            "overall_mean": m["overall"],
                            "overall_std": sdv["overall"],
                            "Head": m["Head"], "Tail": m["Tail"],
                            "tail_std": sdv["Tail"],
                            "raw_gap": m["fairness_gap"],
                            "skill_gap": m["skill_gap"],
                            # base-rate INVARIANT metrics: if a Tail F1 gain is
                            # genuine ranking skill it must show up here too.
                            # If AUC is flat while F1 moves, the "gain" is only
                            # the operating point sliding toward 2p/(1+p).
                            **{k: m.get(k) for k in
                               ("Mid", "Head_auc", "Mid_auc", "Tail_auc",
                                "gap_auc", "Head_tpr", "Tail_tpr", "gap_tpr",
                                "Head_bal", "Tail_bal", "gap_bal",
                                "Head_prec", "Tail_prec")}}) + "\n")
        print("\nRead the 'delta' column: it is the graph's effect at MATCHED")
        print("capacity. Negative at every depth = the graph never helps.")
        print("If delta becomes more negative with depth, over-smoothing is")
        print("confirmed as the mechanism. (Older note, kept for reference:)")
        print("confirmed as the mechanism. Depth 0 is the no-graph control, which")
        print("uses the SAME number of dense layers so capacity is matched.")
        return

    if args.graph_compare:
        print(f"{'Model':18s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} "
              f"{'Tail':>6s} | {'RawGap':>8s} | {'SkillGap':>8s}")
        print("-" * 74)
        gname = f"GNN-{args.gnn} (graph)"
        seeds_g = list(range(args.seed, args.seed + max(1, args.seeds)))
        for ug, name in [(True, gname), (False, "No-GNN (TCN only)")]:
            acc = {k: [] for k in ("overall","Head","Mid","Tail",
                                   "fairness_gap","skill_gap")}
            for sd in seeds_g:
                net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                            use_graph=ug, rounds=args.rounds, seed=sd,
                            pos_weight=pos_weight, gnn_type=args.gnn,
                            fair=args.fair,
                            dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN,
                            gnn_layers=args.gnn_layers, alpha=args.alpha,
                            ode_source=args.ode_source)
                th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
                r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
                for k in acc: acc[k].append(r[k])
            m  = {k: float(np.mean(v)) for k, v in acc.items()}
            sd_= {k: float(np.std(v))  for k, v in acc.items()}
            if len(seeds_g) > 1:
                print(f"{name:18s} | {m['overall']:5.2f}±{sd_['overall']:4.2f} | "
                      f"{m['Head']:6.2f} {m['Mid']:6.2f} "
                      f"{m['Tail']:5.2f}±{sd_['Tail']:4.2f} | "
                      f"{m['fairness_gap']:8.2f} | {m['skill_gap']:8.2f}")
            else:
                print(f"{name:18s} | {m['overall']:7.2f} | {m['Head']:6.2f} "
                      f"{m['Mid']:6.2f} {m['Tail']:6.2f} | "
                      f"{m['fairness_gap']:8.2f} | {m['skill_gap']:8.2f}")
            if args.save:
                import json
                with open(args.save, "a") as fh:
                    fh.write(json.dumps({
                        "city": args.city, "mode": "graph_compare", "model": name,
                        "gnn": args.gnn, "seeds": len(seeds_g),
                        "rounds": args.rounds, "adathr": bool(args.adathr),
                        "overall_mean": m["overall"], "overall_std": sd_["overall"],
                        "Head": m["Head"], "Mid": m["Mid"], "Tail": m["Tail"],
                        "tail_std": sd_["Tail"], "raw_gap": m["fairness_gap"],
                        "skill_gap": m["skill_gap"]}) + "\n")
        print("\nCompare Overall: GNN vs No-GNN. NOTE single-seed runs of this")
        print("comparison are unstable; use --seeds > 1 before drawing conclusions.")
        return

    defenses = (["fedavg","trimmed","krum","vouch","strust","strust2"]
                if args.compare else [args.defense])
    # ------- METRIC-TAXONOMY experiment: the theory's decisive test --------- #
    if args.metrics:
        accum = {}
        seeds = list(range(args.seed, args.seed + max(1, args.seeds)))
        for sd in seeds:
            net = train(clients, A_full, args.defense, args.attack, mal_ids,
                        rounds=args.rounds, seed=sd, pos_weight=pos_weight,
                        gnn_type=args.gnn, fair=args.fair, in_ch=IN,
                        gnn_layers=args.gnn_layers, alpha=args.alpha,
                            ode_source=args.ode_source)
            th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
            r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
            for k, v in r.items():
                if isinstance(v, float):
                    accum.setdefault(k, []).append(v)
        M = {k: float(np.mean(v)) for k, v in accum.items()}
        S = {k: float(np.std(v)) for k, v in accum.items()}
        print(f"\nMETRIC TAXONOMY  —  {args.city.upper()}  "
              f"({len(seeds)} seed(s), same model & predictions)")
        print(f"{'Metric':22s} {'Head':>8s} {'Mid':>8s} {'Tail':>8s} "
              f"{'GAP (H-T)':>11s}")
        print("-" * 62)
        print("BASE-RATE DEPENDENT  (confounded — gap inflated by base rates)")
        for key, name in [("f1", "  Macro-F1"), ("prec", "  Precision"),
                          ("acc", "  Accuracy")]:
            print(f"{name:22s} {M['Head_'+key]:8.2f} {M['Mid_'+key]:8.2f} "
                  f"{M['Tail_'+key]:8.2f} {M['gap_'+key]:11.2f}")
        print("\nBASE-RATE INVARIANT  (unconfounded — reflects true skill)")
        for key, name in [("auc", "  AUC-ROC"), ("bal", "  Balanced acc."),
                          ("tpr", "  Recall / TPR")]:
            print(f"{name:22s} {M['Head_'+key]:8.2f} {M['Mid_'+key]:8.2f} "
                  f"{M['Tail_'+key]:8.2f} {M['gap_'+key]:11.2f}")
        print("\nCORRECTED (skill-normalised F1)")
        print(f"{'  Skill-F1':22s} {M['Head_skill']:8.2f} {M['Mid_skill']:8.2f} "
              f"{M['Tail_skill']:8.2f} {M['skill_gap']:11.2f}")
        ratio = abs(M['gap_f1']) / max(abs(M['gap_auc']), 1e-9)
        print(f"\n=> The SAME predictions yield an F1 gap of {M['gap_f1']:.1f} but an "
              f"AUC gap of only {M['gap_auc']:.2f}")
        print(f"   ({ratio:.0f}x larger). The disparity is a property of the METRIC,")
        print("   not of the model's skill.")
        if args.save:
            import json
            row = {"city": args.city, "mode": "metrics", "seeds": len(seeds),
                   "gnn": args.gnn, "defense": args.defense}
            for k in ["Head_f1","Mid_f1","Tail_f1","gap_f1","Head_prec","Tail_prec",
                      "gap_prec","Head_acc","Tail_acc","gap_acc","Head_auc","Mid_auc",
                      "Tail_auc","gap_auc","Head_bal","Tail_bal","gap_bal","Head_tpr",
                      "Tail_tpr","gap_tpr","Head_skill","Tail_skill","skill_gap"]:
                if k in M: row[k] = round(M[k], 3)
            with open(args.save, "a") as fh: fh.write(json.dumps(row) + "\n")
            print(f"[saved metric-taxonomy row to {args.save}]")
        return

    # ---------------- multi-seed run with mean +/- std and a t-test --------- #
    if args.seeds > 1:
        seeds = list(range(args.seed, args.seed + args.seeds))
        store = {d: {"overall": [], "gap": [], "skill_gap": [],
                     "auc": [], "lift": [], "over_base": []} for d in defenses}
        for sd in seeds:
            for d in defenses:
                net = train(clients, A_full, d, args.attack, mal_ids,
                            rounds=args.rounds, seed=sd, pos_weight=pos_weight,
                            gnn_type=args.gnn, fair=args.fair,
                            dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN,
                            gnn_layers=args.gnn_layers, alpha=args.alpha,
                            ode_source=args.ode_source)
                th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
                r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
                store[d]["overall"].append(r["overall"])
                store[d]["gap"].append(r["fairness_gap"])
                store[d]["skill_gap"].append(r["skill_gap"])
                store[d]["auc"].append(r["auc"])
                store[d]["lift"].append(r["ap_lift"])
                store[d]["over_base"].append(r["f1_over_baseline"])
        print(f"\n{args.seeds} seeds | mean +/- std")
        print(f"{'Defense':9s} | {'Overall':>13s} | {'AUC':>12s} | {'AP lift':>11s} "
              f"| {'F1-baseline':>12s} | {'SkillGap':>12s}")
        print("-" * 88)
        def ms(v): return f"{np.mean(v):6.2f}±{np.std(v):5.2f}"
        for d in defenses:
            s_ = store[d]
            print(f"{d:9s} | {ms(s_['overall']):>13s} | "
                  f"{np.mean(s_['auc']):6.3f}±{np.std(s_['auc']):5.3f} | "
                  f"{np.mean(s_['lift']):5.2f}±{np.std(s_['lift']):4.2f} | "
                  f"{ms(s_['over_base']):>12s} | {ms(s_['skill_gap']):>12s}")
        # paired t-test: our defense vs the best standard baseline defense
        try:
            from scipy import stats
            ours = "strust2" if "strust2" in store else defenses[-1]
            base = max([d for d in defenses if d not in ("strust", "strust2")],
                       key=lambda d: np.mean(store[d]["overall"]))
            a = np.array(store[ours]["overall"]); b = np.array(store[base]["overall"])
            if np.allclose(a, b):
                t, p = 0.0, 1.0
            else:
                t, p = stats.ttest_rel(a, b)
            print(f"\nPaired t-test  {ours} vs {base} (Overall F1): "
                  f"t={t:.3f}, p={p:.4f} "
                  f"{'(significant, p<0.05)' if p < 0.05 else '(NOT significant)'}")
        except Exception as e:
            print(f"\n(t-test unavailable: {e})")
        if args.save:
            import json
            with open(args.save, "a") as fh:
                for d in defenses:
                    s_ = store[d]
                    fh.write(json.dumps({
                        "city": args.city, "attack": args.attack, "gnn": args.gnn,
                        "defense": d, "seeds": args.seeds, "rounds": args.rounds,
                        "adathr": bool(args.adathr), "horizon": args.horizon,
                        "overall_mean": float(np.mean(s_["overall"])),
                        "overall_std": float(np.std(s_["overall"])),
                        "auc_mean": float(np.mean(s_["auc"])),
                        "auc_std": float(np.std(s_["auc"])),
                        "ap_lift": float(np.mean(s_["lift"])),
                        "f1_over_baseline": float(np.mean(s_["over_base"])),
                        "skill_gap_mean": float(np.mean(s_["skill_gap"])),
                        "skill_gap_std": float(np.std(s_["skill_gap"])),
                        "raw_gap_mean": float(np.mean(s_["gap"])),
                    }) + "\n")
            print(f"[saved {len(defenses)} rows to {args.save}]")
        print("\nAUC>0.5 and AP lift>1 mean the model has REAL skill above chance.")
        print("F1-baseline = macro-F1 minus the trivial 'always predict crime' F1.")
        return

    print(f"{'Defense':9s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} {'Tail':>6s} "
          f"| {'RawGap':>7s} | {'H-skill':>7s} {'T-skill':>7s} | {'SkillGap':>8s}")
    print("-" * 86)
    for d in defenses:
        net = train(clients, A_full, d, args.attack, mal_ids,
                    rounds=args.rounds, seed=args.seed, pos_weight=pos_weight,
                    gnn_type=args.gnn, fair=args.fair,
                    dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN,
                    gnn_layers=args.gnn_layers, alpha=args.alpha,
                            ode_source=args.ode_source)
        th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
        r = evaluate(net, Xte, Yte, A_full, tags, mc=args.mc, thr=th)
        print(f"{d:9s} | {r['overall']:7.2f} | {r['Head']:6.2f} {r['Mid']:6.2f} "
              f"{r['Tail']:6.2f} | {r['fairness_gap']:7.2f} | {r['Head_skill']:7.1f} "
              f"{r['Tail_skill']:7.1f} | {r['skill_gap']:8.2f}")
    print(f"\nSKILL CHECK  AUC={r['auc']:.3f} (0.5=no skill) | AP lift={r['ap_lift']:.2f}x "
          f"(1=chance) | trivial-baseline F1={r['baseline_f1']:.1f} | "
          f"model-minus-baseline={r['f1_over_baseline']:+.1f}")
    print("\nRawGap is biased: max F1 = 2p/(1+p) scales with a region's base rate,")
    print("so sparse (tail) regions can NEVER reach head-level raw F1.")
    print("SkillGap = % of each group's ATTAINABLE performance -> the fair comparison.")
    print("SkillGap near 0 means the model serves poor and rich regions equally well.")


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        # Executed by pasting this file into a notebook cell.  argparse would
        # choke on Jupyter's own -f kernel.json argument and print a confusing
        # usage dump, so say what actually went wrong instead.
        print("=" * 70)
        print("This file was RUN as a notebook cell instead of being SAVED.")
        print("Add this as the FIRST line of this cell, then press Enter so")
        print("the docstring starts on line 2:")
        print()
        print("    %%writefile robust_fair_gnn.py")
        print()
        print("Re-run: you should see only 'Writing robust_fair_gnn.py'.")
        print("Then run the script with  !python robust_fair_gnn.py ...")
        print("=" * 70)
    else:
        main()


In [ ]:
%%writefile preprocess_chicago.py
"""
Build the Chicago (2015) FedCrime dataset from the City of Chicago open-data
portal, producing ``data/chi_crime.csv`` with the schema the pipeline expects:

    date_occ, crime_type_id, neighborhood_id

The eight crime categories and their ids follow the paper's Chicago ordering:

    0 robbery   1 battery    2 deceptive_practice  3 burglary
    4 assault   5 theft      6 criminal_damage     7 narcotics

Two input modes
---------------
1. Direct API (default): pages the Socrata endpoint for the eight categories in
   2015. Requires network access on the machine you run this on.

       python scripts/preprocess_chicago.py --out data/chi_crime.csv

2. Local CSV: if you've downloaded the full "Crimes - 2001 to Present" CSV
   (columns include ``Date``, ``Primary Type``, ``Community Area``, ``Year``),
   point at it and skip the network:

       python scripts/preprocess_chicago.py --csv Crimes_-_2001_to_Present.csv \
           --out data/chi_crime.csv

Category counts should closely match the paper (theft ~57k, battery ~49k,
criminal damage ~29k, narcotics ~24k, assault ~17k, deceptive ~16k,
burglary ~13k, robbery ~10k).
"""
from __future__ import annotations

import argparse
import time

import pandas as pd

# paper Chicago primary_type -> crime_type_id
PRIMARY_TO_ID = {
    "ROBBERY": 0,
    "BATTERY": 1,
    "DECEPTIVE PRACTICE": 2,
    "BURGLARY": 3,
    "ASSAULT": 4,
    "THEFT": 5,
    "CRIMINAL DAMAGE": 6,
    "NARCOTICS": 7,
}
RESOURCE = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"


def from_api(year: int = 2015) -> pd.DataFrame:
    from urllib.parse import urlencode
    types = "','".join(PRIMARY_TO_ID)
    where = f"year={year} AND primary_type IN('{types}')"
    rows, offset, page = [], 0, 50000
    while True:
        # URL-encode the query so spaces/quotes in $where don't break the URL
        q = urlencode({"$select": "date,primary_type,community_area",
                       "$where": where, "$limit": page, "$offset": offset})
        url = f"{RESOURCE}?{q}"
        chunk = pd.read_json(url)
        if chunk.empty:
            break
        rows.append(chunk)
        offset += page
        print(f"  fetched {offset} rows...")
        time.sleep(0.5)
    return pd.concat(rows, ignore_index=True)


def from_csv(path: str, year: int = 2015) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=["Date", "Primary Type",
                                    "Community Area", "Year"])
    df = df[df["Year"] == year]
    df = df[df["Primary Type"].isin(PRIMARY_TO_ID)]
    return df.rename(columns={"Date": "date", "Primary Type": "primary_type",
                              "Community Area": "community_area"})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=None, help="local Chicago crimes CSV")
    ap.add_argument("--year", type=int, default=2015)
    ap.add_argument("--out", default="data/chi_crime.csv")
    args = ap.parse_args()

    df = from_csv(args.csv, args.year) if args.csv else from_api(args.year)

    df = df.dropna(subset=["community_area", "primary_type", "date"])
    df["crime_type_id"] = df["primary_type"].str.upper().map(PRIMARY_TO_ID)
    df = df.dropna(subset=["crime_type_id"])
    df["crime_type_id"] = df["crime_type_id"].astype(int)
    # community areas 1..77 -> neighborhood_id 0..76
    df["neighborhood_id"] = (pd.to_numeric(df["community_area"], errors="coerce")
                             .astype("Int64") - 1)
    df = df.dropna(subset=["neighborhood_id"])
    df["date_occ"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

    out = df[["date_occ", "crime_type_id", "neighborhood_id"]].copy()
    out.to_csv(args.out, index=False)
    print(f"\nWrote {len(out)} rows to {args.out}")
    print(f"regions: {out.neighborhood_id.nunique()}  "
          f"categories: {sorted(out.crime_type_id.unique())}")
    print(out.crime_type_id.value_counts().sort_index())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sensitivity.py
"""
Robustness of the literature-audit correlation  (answers reviewer W2 and W3)
================================================================================
W2: base rates are ESTIMATED from published crime totals under a Poisson
    assumption.  If the headline correlation depends on that assumption, the
    claim is fragile.  We therefore recompute it under six different
    assumptions, including ones that make no distributional assumption at all.

W3: n is small (6 models x 5 area groups).  A Pearson r on 5 points per model
    is easy to over-read, so we add a permutation test and a bootstrap CI
    rather than reporting r alone.

Run:  python sensitivity.py
"""
from __future__ import annotations
import numpy as np

rng = np.random.default_rng(0)

GROUPS = ["Very Small", "Small", "Medium", "Large", "Very Large"]
N_COMMUNITIES = np.array([13, 17, 15, 18, 14])
N_CRIMES = np.array([18070, 47813, 52979, 78933, 63409])
CPC = N_CRIMES / N_COMMUNITIES              # crimes per community per year
N_CAT, N_DAYS = 4, 365

MACRO_F1 = {
    "DeepCrime":       [0.24, 0.30, 0.33, 0.38, 0.42],
    "MiST":            [0.18, 0.22, 0.28, 0.31, 0.35],
    "CrimeForecaster": [0.20, 0.39, 0.38, 0.47, 0.43],
    "HAGEN":           [0.25, 0.36, 0.37, 0.42, 0.41],
    "ST-HSL":          [0.39, 0.37, 0.29, 0.32, 0.38],
    "AIST":            [0.46, 0.50, 0.48, 0.52, 0.61],
}
MICRO_F1 = {
    "DeepCrime":       [0.36, 0.41, 0.55, 0.56, 0.59],
    "MiST":            [0.21, 0.34, 0.39, 0.42, 0.45],
    "CrimeForecaster": [0.23, 0.45, 0.45, 0.53, 0.51],
    "HAGEN":           [0.27, 0.39, 0.41, 0.45, 0.44],
    "ST-HSL":          [0.44, 0.48, 0.43, 0.47, 0.60],
    "AIST":            [0.54, 0.60, 0.68, 0.77, 0.73],
}

lam = CPC / (N_DAYS * N_CAT)                # mean events per cell per day

# ---------------------------------------------------------------- W2 variants
def nb_rate(lam, k):
    """Negative-binomial P(at least one event); k = dispersion. k->inf = Poisson.
    Crime clusters in time, so real data is OVER-dispersed relative to Poisson."""
    return 1.0 - (k / (k + lam)) ** k

ASSUMPTIONS = {
    "Poisson (as submitted)":        1 - np.exp(-lam),
    "Neg-binomial k=5 (clustered)":  nb_rate(lam, 5.0),
    "Neg-binomial k=1 (very clustered)": nb_rate(lam, 1.0),
    "Raw rate, no distribution":     lam,
    "Log crime volume":              np.log(CPC),
    "RANK of crime volume only":     np.argsort(np.argsort(CPC)).astype(float),
}

def pearson(a, b): return float(np.corrcoef(a, b)[0, 1])

def spearman(a, b):
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    return pearson(ra, rb)

print("=" * 78)
print("W2 — DOES THE RESULT DEPEND ON THE POISSON ASSUMPTION?")
print("=" * 78)
print("Mean correlation between reported F1 and the base-rate proxy, over 6 models\n")
print(f"{'assumption for base rate':>34} {'macro-F1':>10} {'micro-F1':>10}")
print("-" * 58)
for name, x in ASSUMPTIONS.items():
    ma = np.mean([pearson(x, v) for v in MACRO_F1.values()])
    mi = np.mean([pearson(x, v) for v in MICRO_F1.values()])
    print(f"{name:>34} {ma:10.3f} {mi:10.3f}")
print("\nThe last two rows use NO distributional assumption at all -- 'rank of")
print("crime volume' only needs the ORDER of the groups, which is given directly")
print("in the source papers. If the finding survives there, it does not depend")
print("on how we estimated the base rate.")

# ---------------------------------------------------------------- W3 inference
print("\n" + "=" * 78)
print("W3 — IS THE CORRELATION REAL, GIVEN ONLY 5 POINTS PER MODEL?")
print("=" * 78)

x = ASSUMPTIONS["Poisson (as submitted)"]

def perm_p(x, y, n=200_000):
    """Exact-ish permutation test: how often does a random re-ordering of the
    5 groups produce a correlation at least this strong?"""
    obs = abs(pearson(x, y))
    idx = np.array([rng.permutation(len(y)) for _ in range(n)])
    perms = np.asarray(y)[idx]
    xm = x - x.mean()
    num = perms @ xm
    den = np.sqrt(((perms - perms.mean(1, keepdims=True)) ** 2).sum(1) * (xm ** 2).sum())
    return float(np.mean(np.abs(num / den) >= obs - 1e-12))

print(f"\n{'model':>18} {'r (macro)':>10} {'perm p':>9}   {'r (micro)':>10} {'perm p':>9}")
print("-" * 64)
pa_all, pi_all = [], []
for m in MACRO_F1:
    ya, yi = np.array(MACRO_F1[m]), np.array(MICRO_F1[m])
    ra, ri = pearson(x, ya), pearson(x, yi)
    pa, pi = perm_p(x, ya), perm_p(x, yi)
    pa_all.append(pa); pi_all.append(pi)
    print(f"{m:>18} {ra:10.3f} {pa:9.4f}   {ri:10.3f} {pi:9.4f}")

# pool the 6 models: Fisher's method on the micro-F1 p-values
from math import log
chi2 = -2 * sum(log(max(p, 1e-12)) for p in pi_all)
df = 2 * len(pi_all)
# survival function of chi2 without scipy
def chi2_sf(c, k):
    if k % 2: raise ValueError
    m = k // 2
    t = np.exp(-c / 2); s = t
    for i in range(1, m):
        t *= (c / 2) / i; s += t
    return float(s)
print(f"\nFisher combined test over the 6 models (micro-F1): "
      f"chi2 = {chi2:.1f}, df = {df}, p = {chi2_sf(chi2, df):.2e}")

# pooled correlation with bootstrap CI over MODELS (the unit of generalisation)
names = list(MICRO_F1)
def pooled(sample):
    return float(np.mean([pearson(x, MICRO_F1[n]) for n in sample]))
obs = pooled(names)
boot = [pooled(list(rng.choice(names, len(names), replace=True))) for _ in range(20000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Pooled micro-F1 correlation = {obs:+.3f}   "
      f"95% bootstrap CI over models [{lo:+.3f}, {hi:+.3f}]")

print("\n" + "=" * 78)
print("HOW TO REPORT THIS")
print("  * Do not lead with a single Pearson r on 5 points.")
print("  * Lead with the RANK-based result: it needs only the ordering of the")
print("    groups, which the source papers state directly, so it is assumption-free.")
print("  * Report the permutation p per model, the Fisher combined p, and the")
print("    bootstrap CI across models -- models are the unit of generalisation.")
print("  * ST-HSL is the negative control and should be reported, not hidden.")
print("=" * 78)


In [ ]:
%%writefile auc_check.py
"""
Does the plain-GCN Tail F1 gain reflect real ranking skill, or a moved threshold?
================================================================================
Trains ONLY the two configurations in question -- plain GCN at depth 2 and its
capacity-matched no-graph control -- and reports base-rate DEPENDENT metrics
(F1, precision) beside base-rate INVARIANT ones (AUC, balanced accuracy, TPR).

  Tail AUC rises      -> the graph genuinely ranks sparse regions better.
  Tail AUC flat       -> F1 only slid toward its 2p/(1+p) ceiling; the "gain"
                         is the very artifact this project is about.

Imports robust_fair_gnn rather than editing it, so a "Run all" that rewrites
that file cannot silently undo this.

    python auc_check.py --city chicago --depth 2 --seeds 5 --rounds 40
"""
import argparse, json
import numpy as np
import torch

from robust_fair_gnn import (CITY, C, L, load_city, make_windows, build_graph,
                             head_mid_tail, train, evaluate)

ap = argparse.ArgumentParser()
ap.add_argument("--city", default="chicago")
ap.add_argument("--depth", type=int, default=2)
ap.add_argument("--gnn", default="plain")
ap.add_argument("--seeds", type=int, default=5)
ap.add_argument("--rounds", type=int, default=40)
ap.add_argument("--clients", type=int, default=6)
ap.add_argument("--out", default="results/auc_check.json")
a = ap.parse_args()

csv, dcol, start, end = CITY[a.city]
mat, _ = load_city(csv, dcol, start, end)
print(f"{a.city.upper()}: {mat.shape[1]} regions, {mat.shape[0]} days, "
      f"sparsity {100*(1-mat.mean()):.1f}% zeros")

X, Y = make_windows(mat, horizon=1)
IN = X.shape[-1]
n = len(X); ntr = int(n*0.7); nval = int(n*0.1)
Xtr, Ytr = X[:ntr], Y[:ntr]
Xte, Yte = X[ntr+nval:], Y[ntr+nval:]
A_full, _ = build_graph(mat[:ntr+L])
tags = head_mid_tail(mat[:ntr+L])

order = np.argsort(mat[:ntr].sum(axis=(0, 2)))[::-1]
parts = [order[i::a.clients] for i in range(a.clients)]
dens_r = Ytr.mean(axis=(0, 2))
rw = np.clip(dens_r.mean()/(dens_r+1e-6), 1.0, 4.0).astype(np.float32)
gmap = {"Head": 0, "Mid": 1, "Tail": 2}
grp = np.array([gmap[t] for t in tags], dtype=np.int64)
clients = [{"A": A_full[np.ix_(i, i)], "X": Xtr[:, :, i, :], "Y": Ytr[:, i, :],
            "rw": torch.tensor(rw[i]), "grp": torch.tensor(grp[i])}
           for i in parts]
yflat = Ytr.reshape(-1, C); pos = yflat.sum(0)
pos_weight = torch.tensor(np.clip((len(yflat)-pos)/np.maximum(pos, 1), 1.0, 10.0),
                          dtype=torch.float32)

KEYS = ["Head", "Mid", "Tail", "Head_prec", "Tail_prec",
        "Head_auc", "Mid_auc", "Tail_auc", "gap_auc",
        "Head_bal", "Tail_bal", "gap_bal", "Head_tpr", "Tail_tpr", "gap_tpr",
        "overall", "fairness_gap", "skill_gap"]

def run(use_graph):
    acc = {k: [] for k in KEYS}
    for sd in range(a.seeds):
        net = train(clients, A_full, "fedavg", "none", set(),
                    use_graph=use_graph, rounds=a.rounds, seed=sd,
                    pos_weight=pos_weight, gnn_type=a.gnn, in_ch=IN,
                    gnn_layers=a.depth)
        r = evaluate(net, Xte, Yte, A_full, tags)
        for k in acc:
            acc[k].append(r.get(k, float("nan")))
        print(f"   seed {sd} done", flush=True)
    return ({k: float(np.nanmean(v)) for k, v in acc.items()},
            {k: float(np.nanstd(v)) for k, v in acc.items()})

print(f"\ntraining GRAPH ({a.gnn}, depth {a.depth})...")
mg, sg = run(True)
print(f"training CONTROL (no graph, {a.depth} dense layers)...")
mc, sc = run(False)

BAND = [("BASE-RATE DEPENDENT  (can move without real skill)",
         [("Tail F1", "Tail"), ("Head F1", "Head"),
          ("Tail precision", "Tail_prec"), ("Head precision", "Head_prec"),
          ("raw gap", "fairness_gap"), ("skill gap", "skill_gap")]),
        ("BASE-RATE INVARIANT  (real ranking skill)",
         [("Tail AUC", "Tail_auc"), ("Head AUC", "Head_auc"),
          ("AUC gap", "gap_auc"), ("Tail bal-acc", "Tail_bal"),
          ("bal-acc gap", "gap_bal"), ("Tail TPR", "Tail_tpr"),
          ("TPR gap", "gap_tpr")])]

print("\n" + "="*72)
print(f"{a.city.upper()} — {a.gnn} depth {a.depth} vs matched control "
      f"({a.seeds} seeds, {a.rounds} rounds)")
print("="*72)
for title, items in BAND:
    print("\n  " + title)
    print(f"  {'metric':>15} {'control':>14} {'graph':>14} {'delta':>9}")
    print("  " + "-"*56)
    for lab, k in items:
        print(f"  {lab:>15} {mc[k]:8.2f}±{sc[k]:5.2f} {mg[k]:8.2f}±{sg[k]:5.2f} "
              f"{mg[k]-mc[k]:+9.2f}")

d_f1, d_auc = mg["Tail"]-mc["Tail"], mg["Tail_auc"]-mc["Tail_auc"]
print("\n" + "="*72)
print(f"VERDICT   Tail F1 {d_f1:+.2f}   Tail AUC {d_auc:+.2f}")
if d_auc > 2.0:
    print("  -> AUC moved with F1: the graph genuinely ranks tail regions")
    print("     better. This is a real fairness result.")
elif abs(d_auc) <= 2.0 and d_f1 > 5:
    print("  -> F1 jumped while AUC stayed flat: the model is predicting MORE")
    print("     POSITIVES in sparse regions, sliding F1 toward 2p/(1+p) without")
    print("     gaining skill. That is the artifact, reproduced inside our own")
    print("     model. Check Tail precision above: it should have fallen.")
else:
    print("  -> mixed / no clear effect; do not claim either story.")
print("="*72)

import os
os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
json.dump({"city": a.city, "gnn": a.gnn, "depth": a.depth, "seeds": a.seeds,
           "rounds": a.rounds, "graph_mean": mg, "graph_std": sg,
           "control_mean": mc, "control_std": sc}, open(a.out, "w"), indent=1)
print("saved ->", a.out)


In [ ]:
%%writefile threshold_sweep.py
"""
The artifact without any architecture  (answers reviewer W4)
================================================================================
The GNN result shows Tail F1 rising +16.95 while Tail AUC FALLS 4.01.  A
reviewer will ask whether that is a general property of the metric or a quirk
of one architecture at one depth.

This settles it.  We take the plain no-graph model -- no graph, no depth, no
architecture change of any kind -- and simply MOVE THE DECISION THRESHOLD.

  * AUC is threshold-free, so it is mathematically constant across the sweep.
  * F1 is not, so it traces a curve.

The key number: the threshold at which the ungraphed model REPRODUCES the
graph model's Tail F1.  If a single knob reproduces the entire "fairness
improvement" while ranking ability is provably unchanged, then the improvement
was never about the model.

    python threshold_sweep.py --city chicago --seeds 5 --rounds 40
"""
import argparse, json
import numpy as np
import torch

from robust_fair_gnn import (CITY, C, L, load_city, make_windows, build_graph,
                             head_mid_tail, train)
from sklearn.metrics import roc_auc_score

ap = argparse.ArgumentParser()
ap.add_argument("--city", default="chicago")
ap.add_argument("--seeds", type=int, default=5)
ap.add_argument("--rounds", type=int, default=40)
ap.add_argument("--layers", type=int, default=2)
ap.add_argument("--clients", type=int, default=6)
ap.add_argument("--target-f1", type=float, default=40.38,
                help="Tail F1 the graph model reached (Chicago plain d2)")
ap.add_argument("--out", default="results/threshold_sweep.json")
a = ap.parse_args()

csv, dcol, start, end = CITY[a.city]
mat, _ = load_city(csv, dcol, start, end)
X, Y = make_windows(mat, horizon=1)
IN = X.shape[-1]
n = len(X); ntr = int(n*0.7); nval = int(n*0.1)
Xtr, Ytr = X[:ntr], Y[:ntr]
Xte, Yte = X[ntr+nval:], Y[ntr+nval:]
A_full, _ = build_graph(mat[:ntr+L])
tags = head_mid_tail(mat[:ntr+L])
tail = np.array([t == "Tail" for t in tags])

order = np.argsort(mat[:ntr].sum(axis=(0, 2)))[::-1]
parts = [order[i::a.clients] for i in range(a.clients)]
dens = Ytr.mean(axis=(0, 2))
rw = np.clip(dens.mean()/(dens+1e-6), 1.0, 4.0).astype(np.float32)
gmap = {"Head": 0, "Mid": 1, "Tail": 2}
grp = np.array([gmap[t] for t in tags], dtype=np.int64)
clients = [{"A": A_full[np.ix_(i, i)], "X": Xtr[:, :, i, :], "Y": Ytr[:, i, :],
            "rw": torch.tensor(rw[i]), "grp": torch.tensor(grp[i])} for i in parts]
yf = Ytr.reshape(-1, C); pos = yf.sum(0)
pw = torch.tensor(np.clip((len(yf)-pos)/np.maximum(pos, 1), 1.0, 10.0),
                  dtype=torch.float32)

def tail_scores(net):
    """Return (y, prob) for Tail regions only, flattened per category."""
    net.eval()
    with torch.no_grad():
        _, _, _, logit = net(torch.tensor(Xte), torch.tensor(A_full))
        prob = torch.sigmoid(logit).numpy()
    yg = Yte[:, tail, :]; pg = np.nan_to_num(prob[:, tail, :])
    return yg.reshape(-1, C), pg.reshape(-1, C)

def macro_f1(y, p, thr):
    out = []
    for c in range(C):
        yy, pp = y[:, c], (p[:, c] > thr).astype(int)
        tp = ((pp == 1) & (yy == 1)).sum(); fp = ((pp == 1) & (yy == 0)).sum()
        fn = ((pp == 0) & (yy == 1)).sum()
        if yy.sum() in (0, len(yy)): continue
        out.append(0.0 if 2*tp+fp+fn == 0 else 2*tp/(2*tp+fp+fn))
    return 100*float(np.mean(out)) if out else float("nan")

def macro_auc(y, p):
    out = []
    for c in range(C):
        if y[:, c].sum() in (0, len(y)): continue
        s = p[:, c]
        out.append(0.5 if np.allclose(s, s[0]) else roc_auc_score(y[:, c], s))
    return 100*float(np.mean(out)) if out else float("nan")

THR = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1s, aucs, rates = [], [], []
for sd in range(a.seeds):
    net = train(clients, A_full, "fedavg", "none", set(), use_graph=False,
                rounds=a.rounds, seed=sd, pos_weight=pw, gnn_type="plain",
                in_ch=IN, gnn_layers=a.layers)
    y, p = tail_scores(net)
    f1s.append([macro_f1(y, p, t) for t in THR])
    rates.append([100*float((p > t).mean()) for t in THR])
    aucs.append(macro_auc(y, p))          # ONE value: independent of threshold
    print(f"  seed {sd} done", flush=True)

f1m, f1s_ = np.mean(f1s, 0), np.std(f1s, 0)
ratem = np.mean(rates, 0)
aucm, aucsd = float(np.mean(aucs)), float(np.std(aucs))

print("\n" + "="*76)
print(f"{a.city.upper()} — NO GRAPH AT ALL. Only the decision threshold moves.")
print("="*76)
print(f"{'threshold':>10} {'Tail F1':>14} {'says yes':>10} {'Tail AUC':>16}")
print("-"*56)
for t, m, s, r in zip(THR, f1m, f1s_, ratem):
    print(f"{t:>10.2f} {m:8.2f}±{s:5.2f} {r:9.1f}% {aucm:11.2f}±{aucsd:4.2f}")

best = int(np.nanargmax(f1m))
print(f"\nBest Tail F1 from threshold alone: {f1m[best]:.2f} at threshold {THR[best]:.2f}")
print(f"Graph model reached:               {a.target_f1:.2f}")
print(f"AUC across the ENTIRE sweep:       {aucm:.2f} (constant by construction)")
print()
if f1m[best] >= a.target_f1:
    print("RESULT: the ungraphed model MATCHES OR BEATS the graph model's Tail F1")
    print("using nothing but a different threshold, with identical ranking ability.")
    print("The 'fairness improvement' is reproducible with one knob and no model")
    print("change at all. It was never a property of the architecture.")
else:
    print(f"RESULT: threshold alone reaches {f1m[best]:.2f} of the graph model's "
          f"{a.target_f1:.2f}\n({100*f1m[best]/a.target_f1:.0f}% of it), with AUC "
          "unchanged throughout. Most of the\napparent gain is threshold "
          "placement rather than improved prediction.")
print("="*76)

import os
os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
json.dump({"city": a.city, "seeds": a.seeds, "rounds": a.rounds,
           "thresholds": THR.tolist(), "tail_f1_mean": f1m.tolist(),
           "tail_f1_std": f1s_.tolist(), "positive_rate": ratem.tolist(),
           "tail_auc_mean": aucm, "tail_auc_std": aucsd,
           "graph_target_f1": a.target_f1}, open(a.out, "w"), indent=1)
print("saved ->", a.out)


In [ ]:
!wc -l robust_fair_gnn.py preprocess_chicago.py sensitivity.py auc_check.py threshold_sweep.py

## 3. W2 + W3 — no GPU needed, runs in seconds

Recomputes the literature correlation under six different base-rate assumptions,
including two that make **no distributional assumption at all**, then adds
permutation tests and a bootstrap CI across models.

In [ ]:
!python sensitivity.py

## 4. Build the Chicago dataset

Pages the city open-data portal for 2015. A few minutes. Expected counts:
theft ~57k, battery ~49k, criminal damage ~29k, narcotics ~24k, assault ~17k,
deceptive ~16k, burglary ~13k, robbery ~10k.

In [ ]:
import os
if not os.path.exists('data/chi_crime.csv'):
    !python preprocess_chicago.py --out data/chi_crime.csv
else:
    print('data/chi_crime.csv already built')
import pandas as pd
d = pd.read_csv('data/chi_crime.csv')
print('\nCHI rows:', len(d), '| regions:', d.neighborhood_id.nunique())

## 5. Reproducibility gate

If this prints FAIL, stop — nothing below can be trusted.

In [ ]:
!python robust_fair_gnn.py --city chicago --repro-check --gnn plain \
    --gnn-layers 2 --rounds 20 --seed 0

## 6. W4 — the artifact with NO architecture change

Only the decision threshold moves. Watch two things:

* **Tail AUC** — constant down the whole table (it cannot depend on a threshold)
* **best Tail F1 from threshold alone** vs the graph model's **40.38**

If threshold alone reaches 40.38, the "fairness improvement" needed no graph.

In [ ]:
!python threshold_sweep.py --city chicago --seeds $SEEDS --rounds $ROUNDS \
    --target-f1 40.38 --out results/threshold_sweep.json

## 7. Reference — the graph-vs-control comparison

The result W4 is testing against. Skip if you already have it.

In [ ]:
!python auc_check.py --city chicago --depth 2 --seeds $SEEDS --rounds $ROUNDS \
    --out results/auc_check.json

## 8. Save everything

In [ ]:
import os, json, shutil
for f in sorted(os.listdir('results')):
    print(' ', f, os.path.getsize('results/'+f), 'bytes')

# Colab: download.  Kaggle: files in /kaggle/working appear in the Output panel.
try:
    from google.colab import files
    for f in os.listdir('results'):
        files.download('results/'+f)
except ImportError:
    print('\nKaggle: use the Output panel on the right, then Save Version.')